# ALM Data Pipeline Tutorial

This notebook walks through the ALM (Audio Language Model) data pipeline:
1. Read diarized audio manifests
2. Build training windows with quality filters
3. Remove overlapping windows
4. Inspect and visualize results

**No GPU required** — all stages are CPU-only. Install with `uv sync --extra audio_cpu`.

In [1]:
import json
import os

os.environ["RAY_MAX_LIMIT_FROM_API_SERVER"] = "100000"

from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio.alm.alm_data_builder import ALMDataBuilderStage
from nemo_curator.stages.audio.alm.alm_data_overlap import ALMDataOverlapStage
from nemo_curator.stages.audio.alm.alm_manifest_reader import ALMManifestReader
from nemo_curator.stages.audio.alm.alm_manifest_writer import ALMManifestWriterStage

/home/aaftabv/curator/data/Curator/.venv/lib/python3.12/site-packages/cosmos_xenna/pipelines/private/resources.py:38: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


## Step 1: Inspect the input data

The bundled sample has 5 audio manifest entries with diarized segments.

In [2]:
MANIFEST_PATH = os.path.join("..", "..", "..", "tests", "fixtures", "audio", "alm", "sample_input.jsonl")
OUTPUT_DIR = "./alm_tutorial_output"

with open(MANIFEST_PATH) as f:
    entries = [json.loads(line) for line in f if line.strip()]

print(f"Input entries: {len(entries)}")
for i, entry in enumerate(entries):
    n_segments = len(entry.get("segments", []))
    sr = entry.get("audio_sample_rate", "unknown")
    speakers = len({s.get("speaker", "") for s in entry.get("segments", [])})
    print(f"  [{i}] {n_segments} segments, {speakers} speakers, {sr} Hz")

Input entries: 5
  [0] 30 segments, 3 speakers, 16000 Hz
  [1] 34 segments, 2 speakers, 22050 Hz
  [2] 51 segments, 3 speakers, 16000 Hz
  [3] 36 segments, 3 speakers, 48000 Hz
  [4] 48 segments, 4 speakers, 44100 Hz


## Step 2: Run the ALM pipeline

The pipeline reads manifests, builds windows, filters overlaps, and writes output.

We run with **both** backends (Xenna and Ray Data) and compare results and timing.
Both produce identical results; wall-clock time may differ depending on data size and hardware.

In [3]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "alm_output.jsonl")


def build_pipeline() -> Pipeline:
    """Create a fresh pipeline instance (needed per run since the writer appends)."""
    p = Pipeline(name="alm_tutorial", description="Build ALM training windows from diarized manifests")
    p.add_stage(ALMManifestReader(manifest_path=MANIFEST_PATH))
    p.add_stage(
        ALMDataBuilderStage(
            target_window_duration=120.0,
            tolerance=0.1,
            min_sample_rate=16000,
            min_bandwidth=8000,
            min_speakers=2,
            max_speakers=5,
        )
    )
    p.add_stage(ALMDataOverlapStage(overlap_percentage=50))
    p.add_stage(ALMManifestWriterStage(output_path=output_path))
    return p


print(build_pipeline().describe())

2026-04-23 18:02:09.049 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_manifest_reader' to pipeline 'alm_tutorial'
2026-04-23 18:02:09.049 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_data_builder' to pipeline 'alm_tutorial'
2026-04-23 18:02:09.050 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_data_overlap' to pipeline 'alm_tutorial'
2026-04-23 18:02:09.050 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_manifest_writer' to pipeline 'alm_tutorial'


Pipeline: alm_tutorial
Description: Build ALM training windows from diarized manifests
Stages: 4

Stage 1: alm_manifest_reader
  Resources: 1.0 CPUs
  Batch size: 1
Stage 2: alm_data_builder
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required columns: audio_filepath, segments, audio_sample_rate
Stage 3: alm_data_overlap
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required columns: windows
Stage 4: alm_manifest_writer
  Resources: 1.0 CPUs
  Batch size: 1



In [4]:
import time

from nemo_curator.backends.ray_data import RayDataExecutor

ray_client = RayClient()
ray_client.start()

backends = {
    "xenna": XennaExecutor,
    "ray_data": RayDataExecutor,
}

run_results = {}

for name, executor_cls in backends.items():
    if os.path.exists(output_path):
        os.remove(output_path)

    pipeline = build_pipeline()
    executor = executor_cls()

    t0 = time.time()
    pipeline.run(executor)
    elapsed = time.time() - t0

    with open(output_path) as f:
        data = [json.loads(line) for line in f if line.strip()]

    total_windows = sum(len(r.get("filtered_windows", [])) for r in data)
    total_dur = sum(r.get("filtered_dur", 0) for r in data)

    run_results[name] = {
        "time": elapsed,
        "entries": len(data),
        "windows": total_windows,
        "duration": total_dur,
        "data": data,
    }
    print(f"[{name}] {elapsed:.2f}s — {len(data)} entries, {total_windows} windows, {total_dur:.1f}s audio")

2026-04-23 18:02:09.155 | WARNING  | nemo_curator.core.client:start:121 - No monitoring services are running. Please run the `start_prometheus_grafana.py` script from nemo_curator/metrics folder to setup monitoring services separately.
2026-04-23 18:02:09.157 | INFO     | nemo_curator.core.utils:init_cluster:185 - Ray start command: ray start --head --node-ip-address 127.0.1.1 --port 6380 --metrics-export-port 8080 --dashboard-host 127.0.0.1 --dashboard-port 8266 --ray-client-server-port 20000 --temp-dir /tmp/ray --disable-usage-stats --block
2026-04-23 18:02:19,872	INFO utils.py:85 -- Overwriting previous Ray address (127.0.1.1:6381). Running ray.init() on this node will now connect to the new instance at 127.0.1.1:6380. To override this behavior, pass address=127.0.1.1:6381 to ray.init().


2026-04-23 18:02:10,514	INFO usage_lib.py:447 -- Usage stats collection is disabled.
2026-04-23 18:02:10,514	INFO scripts.py:936 -- Local node IP: 127.0.1.1
2026-04-23 18:02:19,871	SUCC scripts.py:975 -- --------------------
2026-04-23 18:02:19,871	SUCC scripts.py:976 -- Ray runtime started.
2026-04-23 18:02:19,871	SUCC scripts.py:977 -- --------------------
2026-04-23 18:02:19,871	INFO scripts.py:979 -- Next steps
2026-04-23 18:02:19,871	INFO scripts.py:982 -- To add another node to this Ray cluster, run
2026-04-23 18:02:19,871	INFO scripts.py:985 --   ray start --address='127.0.1.1:6380'
2026-04-23 18:02:19,871	INFO scripts.py:996 -- To connect to this Ray cluster:
2026-04-23 18:02:19,871	INFO scripts.py:998 -- import ray
2026-04-23 18:02:19,871	INFO scripts.py:999 -- ray.init(_node_ip_address='127.0.1.1')
2026-04-23 18:02:19,871	INFO scripts.py:1013 -- To submit a Ray job using the Ray Jobs CLI:
2026-04-23 18:02:19,871	INFO scripts.py:1014 --   RAY_API_SERVER_ADDRESS='http://127.0.0

2026-04-23 18:02:24.538 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_manifest_reader' to pipeline 'alm_tutorial'
2026-04-23 18:02:24.538 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_data_builder' to pipeline 'alm_tutorial'
2026-04-23 18:02:24.539 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_data_overlap' to pipeline 'alm_tutorial'
2026-04-23 18:02:24.539 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_manifest_writer' to pipeline 'alm_tutorial'
2026-04-23 18:02:24.540 | INFO     | nemo_curator.pipeline.pipeline:build:70 - Planning pipeline: alm_tutorial
2026-04-23 18:02:24.540 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:106 - Decomposing composite stage: alm_manifest_reader
2026-04-23 18:02:24.541 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:120 - Expanded 'alm_manifest_reader' into 2 execution stages
2026-04-23 18:02:24.550 | INFO 

(raylet) Raylet is terminated. Termination is unexpected. Possible reasons include: (1) SIGKILL by the user or system OOM killer, (2) Invalid memory access from Raylet causing SIGSEGV or SIGBUS, (3) Other termination signals. Last 20 lines of the Raylet logs:
    [2026-04-23 18:02:36,909 W 1786529 1786552] (raylet) store.cc:365: Disconnecting client due to connection error with code 2: End of file
    [2026-04-23 18:02:39,216 I 1786529 1786529] (raylet) node_manager.cc:1420: Disconnecting driver, graceful=true, disconnect_type=3 worker_id=01000000ffffffffffffffffffffffffffffffffffffffffffffffff job_id=01000000
    [2026-04-23 18:02:39,216 I 1786529 1786529] (raylet) node_manager.cc:1549: Driver (pid=1779867) is disconnected. worker_id=01000000ffffffffffffffffffffffffffffffffffffffffffffffff job_id=01000000
    [2026-04-23 18:02:39,217 I 1786529 1786529] (raylet) worker_pool.cc:733: Job 01000000 already started in worker pool.
    [2026-04-23 18:02:39,265 I 1786529 1786529] (raylet) wor

2026-04-23 18:02:46.910 | INFO     | nemo_curator.backends.xenna.executor:execute:151 - Pipeline completed successfully with 5 output tasks
(pid=1788231) /home/aaftabv/curator/data/Curator/.venv/lib/python3.12/site-packages/cosmos_xenna/pipelines/private/resources.py:38: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you. [repeated 14x across cluster]
(pid=1788231)   import pynvml [repeated 14x across cluster]
(Stage 04 - ALMManifestWriterStage pid=1788231) Using blocking ray.get inside async actor. This blocks the event loop. Please use `await` on object ref with asyncio.gather if you want to yield execution to the event loop instead. [repeated 10x across cluster]
2026-04-23 18:02:51.936 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'alm_manifest_reader' to pipeline 'alm_tutorial'
2026-04-23 18:02:51.937 

[xenna] 27.39s — 5 entries, 25 windows, 3035.5s audio


2026-04-23 18:14:52.077 | ERROR    | nemo_curator.backends.ray_data.executor:execute:92 - Error during pipeline execution: 


ConnectionError: 

In [ ]:
MATCH_TOL = 0.1

print("\n" + "=" * 60)
print("Backend Comparison")
print("=" * 60)
print(f"{'':>12s}  {'Xenna':>10s}  {'Ray Data':>10s}  {'Match':>6s}")
print(f"{'Time (s)':>12s}  {run_results['xenna']['time']:10.2f}  {run_results['ray_data']['time']:10.2f}  {'':>6s}")
print(
    f"{'Entries':>12s}  {run_results['xenna']['entries']:10d}  {run_results['ray_data']['entries']:10d}"
    f"  {'✓' if run_results['xenna']['entries'] == run_results['ray_data']['entries'] else '✗':>6s}"
)
print(
    f"{'Windows':>12s}  {run_results['xenna']['windows']:10d}  {run_results['ray_data']['windows']:10d}"
    f"  {'✓' if run_results['xenna']['windows'] == run_results['ray_data']['windows'] else '✗':>6s}"
)
print(
    f"{'Audio (s)':>12s}  {run_results['xenna']['duration']:10.1f}  {run_results['ray_data']['duration']:10.1f}"
    f"  {'✓' if abs(run_results['xenna']['duration'] - run_results['ray_data']['duration']) < MATCH_TOL else '✗':>6s}"
)
speedup = run_results["ray_data"]["time"] / run_results["xenna"]["time"]
faster = "xenna" if speedup > 1 else "ray_data"
print(f"\n→ {faster} was {max(speedup, 1 / speedup):.1f}x faster on this dataset")

results = run_results["xenna"]["data"]

## Step 3: Inspect results

In [ ]:
print(f"Output entries: {len(results)}")

total_windows_before = sum(len(r.get("windows", [])) for r in results)
total_windows_after = sum(len(r.get("filtered_windows", [])) for r in results)
total_duration = sum(r.get("filtered_dur", 0) for r in results)

print(f"Windows before overlap filter: {total_windows_before}")
print(f"Windows after overlap filter:  {total_windows_after}")
print(f"Total filtered audio duration: {total_duration:.1f}s ({total_duration / 3600:.2f}h)")
print(
    f"Overlap reduction: {(1 - total_windows_after / total_windows_before) * 100:.0f}%"
    if total_windows_before
    else "N/A"
)

## Step 4: Understand the filtering statistics

In [ ]:
for i, r in enumerate(results):
    stats = r.get("stats", {})
    print(f"Entry [{i}]:")
    print(f"  Total segments: {stats.get('total_segments', 'N/A')}")
    print(f"  Total duration: {stats.get('total_dur', 0):.1f}s")
    print(f"  Lost (low bandwidth):    {stats.get('lost_bw', 0)}")
    print(f"  Lost (low sample rate):  {stats.get('lost_sr', 0)}")
    print(f"  Lost (speaker count):    {stats.get('lost_spk', 0)}")
    print(f"  Lost (window constraints): {stats.get('lost_win', 0)}")
    print(f"  Truncation events: {r.get('truncation_events', 0)}")
    print()

## Step 5: Visualize results

### Window duration distribution and filter effects

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Collect data from results
all_win_durs = []
all_filt_durs = []
for r in results:
    for w in r.get("windows", []):
        segs = w.get("segments", [])
        if segs:
            dur = segs[-1].get("end", 0) - segs[0].get("start", 0)
            all_win_durs.append(dur)
    all_filt_durs.extend(r.get("filtered_dur_list", []))

# 1. Window duration histogram — before vs after overlap filter
ax = axes[0, 0]
if all_win_durs:
    ax.hist(
        all_win_durs, bins=25, color="#4C72B0", alpha=0.5, label=f"Before ({len(all_win_durs)})", edgecolor="white"
    )
if all_filt_durs:
    ax.hist(
        all_filt_durs, bins=25, color="#C44E52", alpha=0.6, label=f"After ({len(all_filt_durs)})", edgecolor="white"
    )
ax.set_xlabel("Window Duration (seconds)")
ax.set_ylabel("Count")
ax.set_title("Window Durations: Before vs After Overlap Filter")
ax.legend()

# 2. Speaker count distribution across filtered windows
ax = axes[0, 1]
speaker_counts = []
for r in results:
    for fw in r.get("filtered_windows", []):
        speakers = {s.get("speaker", "") for s in fw.get("segments", [])}
        speaker_counts.append(len(speakers))
if speaker_counts:
    counts = Counter(speaker_counts)
    labels = sorted(counts.keys())
    ax.bar([str(k) for k in labels], [counts[k] for k in labels], color="#55A868", edgecolor="white")
ax.set_xlabel("Number of Speakers")
ax.set_ylabel("Number of Windows")
ax.set_title("Speaker Count per Filtered Window")

# 3. Filter loss breakdown — stacked bar per entry
ax = axes[1, 0]
entry_labels = [f"Entry {i}" for i in range(len(results))]
loss_bw = [r.get("stats", {}).get("lost_bw", 0) for r in results]
loss_sr = [r.get("stats", {}).get("lost_sr", 0) for r in results]
loss_spk = [r.get("stats", {}).get("lost_spk", 0) for r in results]
loss_win = [r.get("stats", {}).get("lost_win", 0) for r in results]
x = np.arange(len(results))
w = 0.5
ax.bar(x, loss_bw, w, label="Bandwidth", color="#4C72B0")
ax.bar(x, loss_sr, w, bottom=loss_bw, label="Sample Rate", color="#DD8452")
bottoms = [a + b for a, b in zip(loss_bw, loss_sr, strict=True)]
ax.bar(x, loss_spk, w, bottom=bottoms, label="Speaker Count", color="#55A868")
bottoms2 = [a + b for a, b in zip(bottoms, loss_spk, strict=True)]
ax.bar(x, loss_win, w, bottom=bottoms2, label="Window Constraint", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(entry_labels, fontsize=8)
ax.set_ylabel("Segments Lost")
ax.set_title("Filter Loss Breakdown by Entry")
ax.legend(fontsize=8)

# 4. Per-entry window yield — before vs after
ax = axes[1, 1]
wins_before = [len(r.get("windows", [])) for r in results]
wins_after = [len(r.get("filtered_windows", [])) for r in results]
x = np.arange(len(results))
w = 0.35
ax.bar(x - w / 2, wins_before, w, label="Before Overlap Filter", color="#4C72B0", edgecolor="white")
ax.bar(x + w / 2, wins_after, w, label="After Overlap Filter", color="#C44E52", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(entry_labels, fontsize=8)
ax.set_ylabel("Number of Windows")
ax.set_title("Window Yield per Entry")
ax.legend(fontsize=8)

fig.suptitle(
    f"ALM Pipeline — {len(results)} entries, {sum(wins_before)} → {sum(wins_after)} windows", fontsize=13, y=1.01
)
fig.tight_layout()
plt.show()

print(
    f"\nFiltered durations — min: {min(all_filt_durs):.1f}s, max: {max(all_filt_durs):.1f}s, "
    f"mean: {np.mean(all_filt_durs):.1f}s"
    if all_filt_durs
    else "No filtered windows."
)

## Step 6: Experiment with parameters

Try different overlap thresholds and see how they affect the output:

In [ ]:
from nemo_curator.tasks.audio_task import AudioTask

sample_entry = entries[0]
task = AudioTask(data=sample_entry)

builder = ALMDataBuilderStage(
    target_window_duration=120.0,
    tolerance=0.1,
    min_sample_rate=16000,
    min_bandwidth=8000,
    min_speakers=2,
    max_speakers=5,
)
built = builder.process(task)
n_windows = len(built.data.get("windows", []))

print(f"Windows from entry[0]: {n_windows}")
print()

for pct in [0, 25, 50, 75, 100]:
    overlap = ALMDataOverlapStage(overlap_percentage=pct)
    filtered = overlap.process(built)
    n_filtered = len(filtered.data.get("filtered_windows", []))
    print(
        f"  overlap_percentage={pct:3d}: {n_filtered:3d} windows kept ({n_filtered / n_windows * 100:.0f}%)"
        if n_windows
        else "  No windows"
    )

## Cleanup

Shut down the Ray cluster started by `RayClient`.

In [ ]:
ray_client.stop()